**Imports**

Load pandas and pathlib for the Epoch AI data-center tables.


In [ ]:
import pandas as pd
from pathlib import Path

**Set the data folder**

Point `DATA_DIR` at the folder that holds the Epoch CSVs before reading them.


In [ ]:
DATA_DIR = Path("C:\\Users\\attho\\OneDrive\\Desktop\\College\\ACM ATLAS\\data_centers")   # change to your actual path

**Read the six tables**

Load centers, timelines, chip quantities, chillers, cooling towers, and the chip-type lookup.


In [ ]:
centers = pd.read_csv(DATA_DIR / "data_centers.csv")
timelines = pd.read_csv(DATA_DIR / "data_center_timelines.csv")
chips = pd.read_csv(DATA_DIR / "data_center_chip_quantities.csv")
chillers = pd.read_csv(DATA_DIR / "data_center_chillers.csv")
cooling = pd.read_csv(DATA_DIR / "data_center_cooling_towers.csv")
chip_types = pd.read_csv(DATA_DIR / "chip_types.csv")

**Bundle the tables**

Put the six frames in one dictionary so the profile loops below can visit each of them.


In [ ]:
#inspects the csvs
datasets = {
    "centers": centers,
    "timelines": timelines,
    "chips": chips,
    "chillers": chillers,
    "cooling": cooling,
    "chip_types": chip_types
}

**Shapes and columns**

Print each table's row count and column names to confirm the files loaded as expected.


In [ ]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

**Missing-value helper**

Define a summary that lists only columns with missing values, ranked by percent missing.


In [ ]:
# missing value check
def missing_summary(df):
    missing = df.isna().sum()
    percent = (missing / len(df) * 100).round(2)

    return (
        pd.DataFrame({
            "missing_count": missing,
            "missing_percent": percent
        })
        .query("missing_count > 0")
        .sort_values("missing_percent", ascending=False)
    )

**Missingness by table**

Run that summary on every table so sparse fields are visible before any join.


In [ ]:
for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print(missing_summary(df))

**Preview each table**

Print the first rows of every table to see how names and dates are actually stored.


In [ ]:
#relationship between tables 
for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print(df.head())

**Find id and name columns**

List columns whose names contain "id" or "name". Those are the likely join keys.


In [ ]:
for name, df in datasets.items():
    id_cols = [
        col for col in df.columns
        if "id" in col.lower() or "name" in col.lower()
    ]

    print(name, ":", id_cols)

**Center names in each fact table**

Collect the set of data-center names from the center list, the timelines, and the chip quantities.


In [ ]:
#join verification
center_names = set(centers["Name"].dropna())
timeline_names = set(timelines["Data center"].dropna())
chip_center_names = set(chips["Data center"].dropna())

**How many centers each table names**

Print the size of those three sets. A gap here means a later join will drop or fail to match rows.


In [ ]:
print("Centers:", len(center_names))
print("Timeline centers:", len(timeline_names))
print("Chip centers:", len(chip_center_names))

**Timeline names missing from the center list**

Print timeline centers that do not appear in `data_centers.csv`, which would be orphan timeline rows.


In [ ]:
print("\nTimeline centers missing from centers:")
print(timeline_names - center_names)

**Chip-table names missing from the center list**

Print chip-quantity centers that are not in the center list.


In [ ]:
print("\nChip centers missing from centers:")
print(chip_center_names - center_names)

**Centers with no timeline**

Print center-list names that never appear in the timeline table.


In [ ]:
print("\nCenters without timeline records:")
print(center_names - timeline_names)

**Centers with no chip records**

Print center-list names that have no rows in the chip-quantity table.


In [ ]:
print("\nCenters without chip records:")
print(center_names - chip_center_names)

**Chip types in use versus the lookup**

Collect chip-type names from the quantity table and from the chip-type lookup.


In [ ]:
#chip type matching 
used_chips = set(chips["Chip type"].dropna())
known_chips = set(chip_types["Name"].dropna())

**Chip-type counts**

Print how many distinct chip types are used and how many the lookup contains.


In [ ]:
print("Chip types used:", len(used_chips))
print("Chip types in lookup:", len(known_chips))

**Chip types missing from the lookup**

Print used chip types that would fail a join to `chip_types.csv`.


In [ ]:
print("\nUsed chip types missing from chip_types.csv:")
print(used_chips - known_chips)

**Center-list cardinality**

Compare the number of unique center names with the number of center rows.


In [ ]:
#duplicate and cardinality check
print("Unique centers:", centers["Name"].nunique())
print("Center rows:", len(centers))

**Timeline rows per center**

Summarize how many timeline records each data center has.


In [ ]:
print("\nTimeline records per center:")
print(timelines["Data center"].value_counts().describe())

**Chip rows per center**

Summarize how many chip-quantity records each data center has.


In [ ]:
print("\nChip records per center:")
print(chips["Data center"].value_counts().describe())

**Chip-type lookup cardinality**

Compare unique chip-type names with the number of lookup rows.


In [ ]:
print("\nChip types unique:", chip_types["Name"].nunique())
print("Chip type rows:", len(chip_types))

**Duplicate center names**

Count repeated names in the center list. Duplicates would make a name join ambiguous.


In [ ]:
print("Duplicate center names:",
      centers["Name"].duplicated().sum())

**Duplicate chip-type names**

Count repeated names in the chip-type lookup.


In [ ]:
print("Duplicate chip-type names:",
      chip_types["Name"].duplicated().sum())

**Duplicate timeline rows**

Count timeline rows that are exact copies of another row.


In [ ]:
print("Exact duplicate timeline rows:",
      timelines.duplicated().sum())

**Duplicate chip rows**

Count chip-quantity rows that are exact copies of another row.


In [ ]:
print("Exact duplicate chip rows:",
      chips.duplicated().sum())

**Flag missing water use**

Mark timeline rows whose water-use field is empty so the next cell can compare that gap with construction status.


In [ ]:
#specific check on water use
timelines["water_use_missing"] = timelines["Water use (MGD)"].isna()

**Water-use gaps by construction status**

Show, within each construction status, the share of rows missing water use.


In [ ]:
pd.crosstab(
    timelines["Construction status"],
    timelines["water_use_missing"],
    normalize="index"
)

**Attach chip-type attributes**

Left-join the chip-type lookup onto chip quantities. The join must be many-to-one so quantities are not duplicated.


In [ ]:
#chip table combine
chips_enriched = chips.merge(
    chip_types,
    left_on="Chip type",
    right_on="Name",
    how="left",
    validate="many_to_one",
    suffixes=("", "_chip")
)

**Row count before and after the chip join**

Print both shapes. They should match if the lookup did not multiply rows.


In [ ]:
print("Before:", chips.shape)
print("After:", chips_enriched.shape)

**Assert the chip join kept every row**

Stop if the enriched chip table is longer or shorter than the original quantities.


In [ ]:
assert len(chips_enriched) == len(chips)

**Preview the enriched chip table**

Show center, date, chip type, unit count, H100-equivalent, and release date for a few rows.


In [ ]:
print(
    chips_enriched[
        ["Data center", "Date", "Chip type",
         "Number of Units", "H100e", "Release date"]
    ].head()
)

**Parse timeline dates**

Convert timeline dates to datetimes. Unparseable values become missing so they can be counted.


In [ ]:
#temporal eda
timelines["Date"] = pd.to_datetime(
    timelines["Date"],
    errors="coerce"
)

**Parse chip quantity dates**

Convert the date on each chip-quantity row the same way.


In [ ]:
chips_enriched["Date"] = pd.to_datetime(
    chips_enriched["Date"],
    errors="coerce"
)

**Parse the chip timeline field**

Convert the separate timeline timestamp on the enriched chip table.


In [ ]:
chips_enriched["Timeline"] = pd.to_datetime(
    chips_enriched["Timeline"],
    errors="coerce"
)

**Parse chip release dates**

Convert release dates so they can be compared with the date a chip was recorded at a facility.


In [ ]:
chips_enriched["Release date"] = pd.to_datetime(
    chips_enriched["Release date"],
    errors="coerce"
)

**Count unparseable timeline dates**

Print how many timeline dates failed to parse.


In [ ]:
print("Bad timeline dates:",
      timelines["Date"].isna().sum())

**Count unparseable chip dates**

Print how many chip-quantity dates failed to parse.


In [ ]:
print("Bad chip dates:",
      chips_enriched["Date"].isna().sum())

**Count unparseable chip timeline dates**

Print how many of the chip table's timeline timestamps failed to parse.


In [ ]:
print("Bad chip timeline dates:",
      chips_enriched["Timeline"].isna().sum())

**Add a year column**

Take the calendar year from the parsed timeline and chip dates for the counts below.


In [ ]:
timelines["Year"] = timelines["Date"].dt.year
chips_enriched["Year"] = chips_enriched["Date"].dt.year

**Timeline rows by year**

Print how many timeline records fall in each year.


In [ ]:
print("\nTimeline year distribution:")
print(
    timelines["Year"]
    .value_counts()
    .sort_index()
)

**Chip rows by year**

Print how many chip-quantity records fall in each year.


In [ ]:
print("\nChip year distribution:")
print(
    chips_enriched["Year"]
    .value_counts()
    .sort_index()
)

**Distinct facilities per year**

Count unique data centers with at least one timeline row in each year.


In [ ]:
#unique facilities per year
facilities_per_year = (
    timelines
    .groupby("Year")["Data center"]
    .nunique()
)

**Print facilities per year**

Show that yearly facility count.


In [ ]:
print(facilities_per_year)

**First and last observation per facility**

For each data center, record the earliest timeline date, the latest, and how many observations it has.


In [ ]:
#per facility temporal coverage 
temporal_coverage = (
    timelines
    .groupby("Data center")["Date"]
    .agg(
        first_observation="min",
        last_observation="max",
        n_observations="count"
    )
    .reset_index()
)

**Observation span in years**

Turn the gap between the first and last observation into years.


In [ ]:
temporal_coverage["span_years"] = (
    temporal_coverage["last_observation"]
    - temporal_coverage["first_observation"]
).dt.days / 365.25

**Print temporal coverage**

Show the per-facility coverage table, including the span just computed.


In [ ]:
print(temporal_coverage)

**Construction status counts**

Print how often each construction status appears, including missing labels.


In [ ]:
#construction status
print(
    timelines["Construction status"]
    .value_counts(dropna=False)
)

**Construction status by year**

Cross-tabulate year and construction status to see how the mix shifts over time.


In [ ]:
status_by_year = pd.crosstab(
    timelines["Year"],
    timelines["Construction status"]
)

**Print status by year**

Show that cross-tab.


In [ ]:
print(status_by_year)

**Sort timelines for a leakage look**

Order each facility's timeline by date so early and late fields can be compared in the next cell.


In [ ]:
#leakage check 
timeline_sorted = timelines.sort_values(
    ["Data center", "Date"]
)

**Preview fields that might leak future state**

Show construction status, buildings, power, and H100 equivalents on the earliest rows. Columns that already reflect a later total should not be used as historical features.


In [ ]:
timeline_sorted[
    ["Data center", "Date",
     "Construction status",
     "Buildings operational",
     "Power (MW)",
     "H100 equivalents"]
].head(30)

**Note a leakage-suspect column**

Name the "current H100 equivalents" field, which reads as a present-day total rather than a value dated to the timeline row.


In [ ]:
LEAKAGE_SUSPECTS = [
    "Current H100 equivalents (from Data center)"
]

**Set the future-date cutoff**

Use 5 October 2026 as the cutoff for marking timeline rows that are dated after the audit.


In [ ]:
#temporal audit
cutoff = pd.Timestamp("2026-10-05")

**Flag dates after the cutoff**

Mark timeline rows later than that cutoff so future-dated records can be counted and listed.


In [ ]:
timelines["is_future"] = timelines["Date"] > cutoff

**Count future-dated timeline rows**

Print how many rows fall after the cutoff and how many do not.


In [ ]:
print(timelines["is_future"].value_counts())

**Sample the future-dated rows**

Show the earliest rows dated after the cutoff, with center name, date, and construction status.


In [ ]:
print(
    timelines.loc[
        timelines["is_future"],
        ["Data center", "Date", "Construction status"]
    ]
    .sort_values("Date")
    .head(20)
)

**Flag rows on or before the end of 2023**

Mark timeline dates through 31 December 2023. The next cells use this to see which facilities have a historical record.


In [ ]:
timelines["Year"] = timelines["Date"].dt.year
timelines["is_2023_or_earlier"] = timelines["Date"] <= "2023-12-31"

**Flag chips recorded before their release date**

Mark enriched chip rows whose recorded date is earlier than the chip's release date.


In [ ]:
#audit flag
chips_enriched["temporal_inconsistency"] = (
    chips_enriched["Date"] < chips_enriched["Release date"]
)

**2023 coverage by facility**

For each data center, record the first and last timeline date and whether any row is on or before the end of 2023.


In [ ]:
#checking to see what datacenters are compatible with 2023
coverage_2023 = (
    timelines
    .groupby("Data center")
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        has_2023_or_earlier=("is_2023_or_earlier", "max")
    )
    .reset_index()
)

**How many facilities have a 2023-or-earlier row**

Print the count of facilities with and without a timeline record by the end of 2023.


In [ ]:
print(
    coverage_2023["has_2023_or_earlier"]
    .value_counts()
)

**Latest record on or before the end of 2023**

Keep each facility's last timeline row dated no later than 31 December 2023.


In [ ]:
#latest 2023 record
historical_2023 = (
    timelines[
        timelines["Date"] <= pd.Timestamp("2023-12-31")
    ]
    .sort_values("Date")
    .groupby("Data center")
    .tail(1)
)

**Count facilities with a 2023 record**

Print how many data centers have at least one of those historical rows.


In [ ]:
print("Facilities with a record by end of 2023:")
print(historical_2023["Data center"].nunique())

**Show the end-of-2023 snapshot**

Print date, construction status, buildings, and power for each facility's last 2023-or-earlier record.


In [ ]:
print(
    historical_2023[
        [
            "Data center",
            "Date",
            "Construction status",
            "Buildings operational",
            "IT power (MW)",
            "Power (MW)"
        ]
    ].sort_values("Date")
)

**First timeline row per facility**

Keep the earliest dated record for each data center, to see when sites first appear.


In [ ]:
#checks when facilities first appear
first_records = (
    timelines
    .sort_values("Date")
    .groupby("Data center")
    .first()
    .reset_index()
)

**Year of first appearance**

Take the calendar year from that first record.


In [ ]:
first_records["first_year"] = first_records["Date"].dt.year

**Facilities by first-seen year**

Print how many data centers first appear in each year.


In [ ]:
print(
    first_records["first_year"]
    .value_counts()
    .sort_index()
)

**List first records**

Print every facility's first date and construction status, in date order.


In [ ]:
print(
    first_records[
        ["Data center", "Date", "Construction status"]
    ]
    .sort_values("Date")
    .to_string(index=False)
)

**Chip rows dated before release**

Repeat the release-date comparison on a column named for this sanity check, so the failing rows can be listed next.


In [ ]:
#chip sanity check
chips_enriched["predates_release"] = (
    chips_enriched["Date"] <
    chips_enriched["Release date"]
)

**Count chips dated before release**

Print how many chip rows predate their release date, including missing comparisons.


In [ ]:
print(
    chips_enriched["predates_release"]
    .value_counts(dropna=False)
)

**List chips dated before release**

Show the facility, chip type, recorded date, release date, and unit count for those rows.


In [ ]:
print(
    chips_enriched.loc[
        chips_enriched["predates_release"] == True,
        [
            "Data center",
            "Chip type",
            "Date",
            "Release date",
            "Number of Units"
        ]
    ].sort_values("Date")
)

**Centers by country**

Print the country counts. The following cells keep the United States subset used for ATLAS.


In [ ]:
#Spatial relavance check
print(centers["Country"].value_counts(dropna=False))

**Keep United States centers**

Filter the center list to facilities whose country is the United States.


In [ ]:
us_centers = centers[
    centers["Country"].eq("United States")
].copy()

**US versus all-center counts**

Print how many Epoch centers there are in total and how many are in the United States.


In [ ]:
print("All Epoch centers:", len(centers))
print("US centers:", len(us_centers))

**Attach country to each timeline row**

Join the center list's country onto timelines by facility name, without multiplying timeline rows.


In [ ]:
#attach to timeline
timeline_geo = timelines.merge(
    centers[["Name", "Country"]],
    left_on="Data center",
    right_on="Name",
    how="left",
    validate="many_to_one"
)

**Assert the country join kept every timeline row**

Stop if the geographic timeline is a different length from the original timeline table.


In [ ]:
assert len(timeline_geo) == len(timelines)

**US timeline only**

Keep timeline rows whose joined country is the United States.


In [ ]:
#relavant temporal coverage
us_timeline = timeline_geo.loc[
    timeline_geo["Country"] == "United States"
].copy()

**Mark operational US rows**

Treat a US timeline row as operational when its operational-building count is greater than zero.


In [ ]:
us_timeline["is_operational"] = (
    us_timeline["Buildings operational"] > 0
)

**US facilities per year**

Count distinct United States data centers with a timeline row in each year.


In [ ]:
print(
    us_timeline.groupby("Year")["Data center"]
    .nunique()
)

**Recompute the operational flag**

Set the same operational marker again so the yearly diagnostic below does not depend on an earlier edit.


In [ ]:
#"operational" status tag diagnostic
us_timeline["is_operational"] = (
    us_timeline["Buildings operational"] > 0
)

**Operational records by year**

For each year, count distinct US facilities and how many of their timeline rows are marked operational.


In [ ]:
operational_by_year = (
    us_timeline
    .groupby("Year")
    .agg(
        facilities=("Data center", "nunique"),
        operational_records=("is_operational", "sum")
    )
)

**Print the operational-by-year table**

Show that yearly summary.


In [ ]:
print(operational_by_year)

**Distinct operational facilities by year**

Count unique US data centers that have at least one operational row in each year.


In [ ]:
operational_facilities = (
    us_timeline[
        us_timeline["is_operational"]
    ]
    .groupby("Year")["Data center"]
    .nunique()
)

**Print operational facilities by year**

Show that series.


In [ ]:
print(operational_facilities)